# Walmart Store Sales Forecasting — N-BEATS

**ლოგირება:** Weights & Biases — პროექტი `ML-Final`, group `NBEATS_Training`.

**რა არის N-BEATS** (Oreshkin et al., *"N-BEATS: Neural basis expansion analysis for interpretable time series forecasting"*, ICLR 2020):
სტეკი ბლოკებისგან შედგება, თითოეული ორ ნაწილს სწავლობს — **backcast** (ეხმარება წინა ბლოკს დარჩენილი სიგნალის ამოცნობაში, doubly residual) და **forecast** (მომავალი პროგნოზი). ბლოკები ჯამდება. ორიგინალი paper უნივარიატულია (მხოლოდ საკუთარი წარსული), მაგრამ ჩვენ ვამატებთ **IsHoliday**-ს, როგორც ცნობილ მომავალ ეგზოგენურ ფიჩერს (NBEATSx-style), რადგან Walmart-ის გაყიდვები ძლიერად არის დამოკიდებული სადღესასწაულო კვირებზე.

Run-ების სტრუქტურა:
1. `NBEATS_Preprocessing` — Date × (Store, Dept) მატრიცის აგება (იგივე, რაც DLinear-ში — ყველა სერია ინარჩუნებს სრულ სიგრძეს, არაფერი იშლება)
2. `NBEATS_candidate_{i}` — n_blocks / layer_width / epochs sweep
3. `NBEATS_Final_Pipeline` — საბოლოო მოდელი მთელ ისტორიაზე, W&B Artifact-ად შენახული

In [2]:
!pip install -q wandb

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import wandb

wandb.login()

WANDB_PROJECT = 'ML-Final'
GROUP = 'NBEATS_Training'
NB_VERSION = 'v1'

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: amakh23 (dshon23-free-university-of-tbilisi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
wandb.login(relogin=True)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: ERROR Invalid API key: API key must have 40+ characters, has 36.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: amakh23 (dshon23-free-university-of-tbilisi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import wandb
print(wandb.api.default_entity)

dshon23-free-university-of-tbilisi


In [ ]:
import wandb
api = wandb.Api()
print(api.viewer.username)

amakh23


In [5]:
import numpy as np
import pandas as pd

BASE = '/content/drive/MyDrive/ML Final/Data/Raw/walmart-recruiting-store-sales-forecasting/'

train = pd.read_csv(BASE + 'train.csv')
test = pd.read_csv(BASE + 'test.csv')
features = pd.read_csv(BASE + 'features.csv')
stores = pd.read_csv(BASE + 'stores.csv')

for d in (train, test, features):
    d['Date'] = pd.to_datetime(d['Date'])

RAW_COLS = ['Store', 'Dept', 'Date', 'IsHoliday']
VAL_CUTOFF = '2012-08-17'


def wmae(y_true, y_pred, is_holiday):
    w = np.where(np.asarray(is_holiday).astype(bool), 5, 1)
    return np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w)


print(train.shape, test.shape, features.shape, stores.shape)

(421570, 5) (115064, 4) (8190, 12) (45, 3)


## Run 1 — Preprocessing

იგივე Date × (Store, Dept) მატრიცა, რაც DLinear-ში. **მნიშვნელოვანი:** ეს pivot ავტომატურად ინახავს ყველა სერიას სრული სიგრძით (ადრეული/მოკლე სერიები 0-ით ივსება), ამიტომ short-series-ის ცალკე გაფილტვრა საჭირო არაა — არც ერთი Store-Dept წყვილი არ იკარგება.

დამატებით ვაშენებთ **holiday ვექტორს** — Date -> IsHoliday (ცნობილი მომავალი ეგზოგენური ფიჩერი, ერთნაირია ყველა store-ისთვის მოცემულ თარიღზე).

In [6]:
run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='NBEATS_Preprocessing',
                 job_type='preprocessing',
                 config={'negative_sales_strategy': 'clip_to_zero',
                         'missing_strategy': 'fill_zero (late-starting / gappy series)',
                         'normalization': 'per-series standardization, std clipped to >= 1',
                         'exogenous': 'IsHoliday (known future covariate)'})

train['Weekly_Sales'] = train['Weekly_Sales'].clip(lower=0)

wide = train.pivot_table(index='Date', columns=['Store', 'Dept'], values='Weekly_Sales')
wide = wide.sort_index()
wide.index.name = 'Date'

missing_pct = float(wide.isna().mean().mean())
wide = wide.fillna(0.0)

holiday_by_date = train.groupby('Date')['IsHoliday'].max()
full_date_range = pd.DatetimeIndex(sorted(set(wide.index) | set(test['Date'].unique())))
holiday_by_date = holiday_by_date.reindex(full_date_range).fillna(False).astype(bool)

run.config['matrix_shape'] = str(wide.shape)
run.config['n_series_kept'] = wide.shape[1]
run.log({'missing_cell_pct': missing_pct, 'n_series': wide.shape[1], 'n_weeks': wide.shape[0]})
print('weeks x series:', wide.shape, '| missing cells:', f'{missing_pct:.1%}', '| n_series kept:', wide.shape[1])
run.finish()

val_dates = wide.index[wide.index >= VAL_CUTOFF]
PRED_LEN_VAL = len(val_dates)
print('validation weeks:', PRED_LEN_VAL)

weeks x series: (143, 3331) | missing cells: 11.5% | n_series kept: 3331


/tmp/ipykernel_552/948486287.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  holiday_by_date = holiday_by_date.reindex(full_date_range).fillna(False).astype(bool)


missing_cell_pct,▁
n_series,▁
n_weeks,▁
missing_cell_pct,0.11497
n_series,3331
n_weeks,143


validation weeks: 11


## N-BEATS იმპლემენტაცია (PyTorch, NBEATSx-style ეგზოგენური holiday input-ით)

- `NBeatsBlock` — FC სტეკი, რომელიც `seq_len` ისტორიას + `pred_len` სიგრძის მომავალ holiday ვექტორს იღებს შესასვლელად; აბრუნებს **backcast** (წარსულის ნაწილი, აკლდება residual-ს) და **forecast**-ს
- `NBeatsNet` — რამდენიმე ბლოკის doubly-residual ჯამი (ისევე, როგორც ორიგინალი paper-ში)
- **channel-shared**: ერთი წონების ნაკრები ყველა Store-Dept სერიაზე (DLinear-ის გამოცდილებით — `individual=True` overfit-დებოდა)
- **exogenous**: IsHoliday მომავალი ჰორიზონტისთვის ცნობილია წინასწარ (Kaggle test.csv-შიც არის) — ლეგალურია მისი გამოყენება

In [7]:
import torch
import torch.nn as nn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print('device:', DEVICE)


class NBeatsBlock(nn.Module):
    def __init__(self, seq_len, pred_len, exog_size=0, layer_width=256, n_layers=4):
        super().__init__()
        input_dim = seq_len + exog_size
        layers = []
        for i in range(n_layers):
            layers += [nn.Linear(input_dim if i == 0 else layer_width, layer_width), nn.ReLU()]
        self.fc = nn.Sequential(*layers)
        self.theta_b = nn.Linear(layer_width, seq_len)
        self.theta_f = nn.Linear(layer_width, pred_len)

    def forward(self, x, exog=None):
        inp = torch.cat([x, exog], dim=-1) if exog is not None else x
        h = self.fc(inp)
        return self.theta_b(h), self.theta_f(h)


class NBeatsNet(nn.Module):
    """Generic N-BEATS: doubly-residual stack of blocks, channel-shared, with exogenous input."""

    def __init__(self, seq_len, pred_len, exog_size=0, n_blocks=4, layer_width=256, n_layers=4):
        super().__init__()
        self.pred_len = pred_len
        self.blocks = nn.ModuleList([
            NBeatsBlock(seq_len, pred_len, exog_size, layer_width, n_layers)
            for _ in range(n_blocks)
        ])

    def forward(self, x, exog=None):
        residual = x
        forecast = torch.zeros(x.shape[0], self.pred_len, device=x.device)
        for block in self.blocks:
            backcast, block_forecast = block(residual, exog)
            residual = residual - backcast
            forecast = forecast + block_forecast
        return forecast

device: cuda


In [8]:
from torch.utils.data import DataLoader, TensorDataset


def make_windows(arr, holiday_arr, seq_len, pred_len):
    """arr: (T, N) sales matrix, holiday_arr: (T,) bool -> X:(n, seq_len, N), Y:(n, pred_len, N), E:(n, pred_len)."""
    xs, ys, es = [], [], []
    for t in range(arr.shape[0] - seq_len - pred_len + 1):
        xs.append(arr[t:t + seq_len])
        ys.append(arr[t + seq_len:t + seq_len + pred_len])
        es.append(holiday_arr[t + seq_len:t + seq_len + pred_len].astype(np.float32))
    return np.stack(xs), np.stack(ys), np.stack(es)


def train_nbeats(arr, holiday_arr, seq_len, pred_len, n_blocks=4, layer_width=256, n_layers=4,
                  epochs=80, batch_size=16, lr=1e-3, log_fn=None):
    X, Y, E = make_windows(arr, holiday_arr, seq_len, pred_len)
    N = arr.shape[1]
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                        torch.tensor(Y, dtype=torch.float32),
                        torch.tensor(E, dtype=torch.float32))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    model = NBeatsNet(seq_len, pred_len, exog_size=1 if False else pred_len,
                       n_blocks=n_blocks, layer_width=layer_width, n_layers=n_layers).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    losses = []
    for ep in range(epochs):
        model.train()
        tot = 0.0
        for xb, yb, eb in dl:
            B = xb.shape[0]
            xb = xb.permute(0, 2, 1).reshape(B * N, seq_len).to(DEVICE)
            yb = yb.permute(0, 2, 1).reshape(B * N, pred_len).to(DEVICE)
            eb = eb.unsqueeze(1).repeat(1, N, 1).reshape(B * N, pred_len).to(DEVICE)
            opt.zero_grad()
            pred = model(xb, eb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            tot += loss.item() * B
        losses.append(tot / len(ds))
        if log_fn:
            log_fn(ep, losses[-1])
        if (ep + 1) % 10 == 0:
            print(f'  epoch {ep + 1:>3}: train MSE {losses[-1]:.4f}')
    return model, losses


def forecast_next(model, arr, exog_future, seq_len):
    """ისტორიის ბოლო seq_len კვირიდან პროგნოზი, ცნობილი მომავალი holiday ვექტორით: (pred_len, N)."""
    model.eval()
    N = arr.shape[1]
    x = torch.tensor(arr[-seq_len:], dtype=torch.float32).permute(1, 0).to(DEVICE)  # (N, seq_len)
    exog = torch.tensor(exog_future, dtype=torch.float32).unsqueeze(0).repeat(N, 1).to(DEVICE)
    with torch.no_grad():
        pred = model(x, exog)
    return pred.cpu().numpy().T


val_actual = train[train['Date'] >= VAL_CUTOFF][['Store', 'Dept', 'Date', 'IsHoliday', 'Weekly_Sales']]


def matrix_val_wmae(pred_matrix):
    """(PRED_LEN_VAL, N) -> WMAE ვალიდაციის რეალურ სტრიქონებზე."""
    pred_df = pd.DataFrame(pred_matrix, index=val_dates, columns=wide.columns)
    pred_long = pred_df.stack([0, 1]).rename('pred').reset_index()
    pred_long.columns = ['Date', 'Store', 'Dept', 'pred']
    merged = val_actual.merge(pred_long, on=['Store', 'Dept', 'Date'], how='left')
    merged['pred'] = merged['pred'].fillna(merged['Weekly_Sales'].mean())
    return wmae(merged['Weekly_Sales'], merged['pred'], merged['IsHoliday'])

## Run 2 — Hyperparameter Search

შესადარებელი პარამეტრები:
- `seq_len` (lookback): 52 / 78 კვირა
- `n_blocks` / `layer_width`: მოდელის capacity (3×128 vs 4×256)
- `epochs`: 60 / 80 / 120 — საკმარისი კონვერგენციისთვის (loss curve wandb-ზე მონიტორინგდება)

ყველა candidate იღებს **IsHoliday**-ს, როგორც ცნობილ მომავალ ეგზოგენურ ფიჩერს — მოდელი წინასწარ „ხედავს" როდის მოდის სადღესასწაულო კვირა, არა მხოლოდ საკუთარ წარსულ გაყიდვებს.

ვალიდაცია: მოდელი ტრენინგდება `< 2012-08-17` მონაცემზე და პროგნოზირებს holdout-ის ყველა კვირას (`pred_len = PRED_LEN_VAL`), WMAE ითვლება რეალურ (Store, Dept, Date) სტრიქონებზე — იგივე სქემა, რაც სხვა არქიტექტურებში.

In [ ]:
tr_wide = wide[wide.index < VAL_CUTOFF]
mu = tr_wide.values.mean(axis=0)
sigma = np.clip(tr_wide.values.std(axis=0), 1.0, None)
norm_tr = (tr_wide.values - mu) / sigma

tr_holiday = holiday_by_date.reindex(tr_wide.index).values
val_holiday_future = holiday_by_date.reindex(val_dates).values.astype(np.float32)

CANDIDATES = [
    dict(seq_len=52, n_blocks=3, layer_width=128, epochs=60, lr=1e-3),
    dict(seq_len=52, n_blocks=4, layer_width=256, epochs=80, lr=1e-3),
    dict(seq_len=78, n_blocks=4, layer_width=256, epochs=80, lr=1e-3),
    dict(seq_len=52, n_blocks=4, layer_width=256, epochs=120, lr=5e-4),
]

best_val, BEST_CFG = np.inf, None
for i, cfg in enumerate(CANDIDATES):
    run = wandb.init(project=WANDB_PROJECT, group=GROUP, name=f'NBEATS_candidate_{i}',
                     job_type='hyperparam_search',
                     config={**cfg, 'pred_len': PRED_LEN_VAL, 'exogenous': 'IsHoliday',
                             'val_scheme': f'train < {VAL_CUTOFF}, predict {PRED_LEN_VAL} weeks ahead'})
    print(f'candidate {i}: {cfg}')
    model, losses = train_nbeats(norm_tr, tr_holiday, pred_len=PRED_LEN_VAL, **cfg,
                                  log_fn=lambda ep, l: run.log({'epoch': ep, 'train_mse': l}))
    pred = forecast_next(model, norm_tr, val_holiday_future, cfg['seq_len']) * sigma + mu
    score = float(matrix_val_wmae(pred))
    run.summary['val_wmae'] = score
    run.finish()
    print(f'  -> val WMAE {score:,.2f}')
    if score < best_val:
        best_val, BEST_CFG = score, cfg

print('Best:', BEST_CFG, f'-> {best_val:,.2f}')

candidate 0: {'seq_len': 52, 'n_blocks': 3, 'layer_width': 128, 'epochs': 60, 'lr': 0.001}
  epoch  10: train MSE 0.6291
  epoch  20: train MSE 0.5748
  epoch  30: train MSE 0.5597
  epoch  40: train MSE 0.5306
  epoch  50: train MSE 0.5181
  epoch  60: train MSE 0.5072


/tmp/ipykernel_543/1391165103.py:66: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  pred_long = pred_df.stack([0, 1]).rename('pred').reset_index()


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇██
train_mse,█▇▆▅▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,59
train_mse,0.50722
val_wmae,1476.8615


  -> val WMAE 1,476.86


candidate 1: {'seq_len': 52, 'n_blocks': 4, 'layer_width': 256, 'epochs': 80, 'lr': 0.001}
  epoch  10: train MSE 0.5883
  epoch  20: train MSE 0.5341
  epoch  30: train MSE 0.5063
  epoch  40: train MSE 0.4895
  epoch  50: train MSE 0.4741
  epoch  60: train MSE 0.4673
  epoch  70: train MSE 0.4612
  epoch  80: train MSE 0.4536


...

  -> val WMAE 1,502.52


candidate 3: {'seq_len': 52, 'n_blocks': 4, 'layer_width': 256, 'epochs': 120, 'lr': 0.0005}
  epoch  10: train MSE 0.6060
  epoch  20: train MSE 0.5630
  epoch  30: train MSE 0.5369
  epoch  40: train MSE 0.5181
  epoch  50: train MSE 0.5133
  epoch  60: train MSE 0.4949
  epoch  70: train MSE 0.4817
  epoch  80: train MSE 0.4820
  epoch  90: train MSE 0.4697
  epoch 100: train MSE 0.4609
  epoch 110: train MSE 0.4591
  epoch 120: train MSE 0.4677


/tmp/ipykernel_543/1391165103.py:66: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  pred_long = pred_df.stack([0, 1]).rename('pred').reset_index()


epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇█
train_mse,█▆▆▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch,119
train_mse,0.46772
val_wmae,1533.28352


  -> val WMAE 1,533.28
Best: {'seq_len': 52, 'n_blocks': 3, 'layer_width': 128, 'epochs': 60, 'lr': 0.001} -> 1,476.86


## Run 3 — Final Pipeline

საბოლოო მოდელი მთელ ისტორიაზე (2010-02-05 … 2012-10-26) ტრენინგდება და პროგნოზირებს Kaggle test-ის **39 კვირას** (2012-11-02 … 2013-07-26), რეალური test.csv-ის holiday ვექტორის გამოყენებით.

`NBEATSForecastPipeline` კლასი, ისევე როგორც DLinear-ში, ინახავს მზა პროგნოზების ცხრილს + fallback საშუალოებს (Store-Dept → Dept → global mean) train-ში უნახავი კომბინაციებისთვის; `predict()` **raw** test dataframe-ს იღებს პირდაპირ. მთელი ობიექტი `cloudpickle`-ით ინახება W&B Artifact-ად (`nbeats_pipeline:latest`) და `best_runs.json`-ში ილინკება, რომ `model_inference.ipynb`-მა შეძლოს არქიტექტურების შედარება.

**საბოლოო N-BEATS val WMAE: 1 476.9** — tree მოდელების დონეზე (XGBoost 1 451.4, LightGBM 1 508.4), naive baseline-ს (2 499) 41%-ით სჯობს.

In [14]:
import cloudpickle

test_dates = pd.DatetimeIndex(np.sort(test['Date'].unique()), name='Date')
PRED_LEN_TEST = len(test_dates)
assert (test_dates[0] - wide.index.max()).days == 7, 'test must start right after train'
test_holiday_future = holiday_by_date.reindex(test_dates).values.astype(np.float32)
print('test horizon:', PRED_LEN_TEST, 'weeks')


class NBEATSForecastPipeline:
    def __init__(self, forecast_long, sd_mean, dept_mean, global_mean):
        self.forecast_long = forecast_long
        self.sd_mean = sd_mean
        self.dept_mean = dept_mean
        self.global_mean = global_mean

    def predict(self, model_input):
        df = model_input.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        out = (df.merge(self.forecast_long, on=['Store', 'Dept', 'Date'], how='left')
                 .merge(self.sd_mean, on=['Store', 'Dept'], how='left')
                 .merge(self.dept_mean, on='Dept', how='left'))
        pred = (out['pred'].fillna(out['SD_Mean'])
                           .fillna(out['Dept_Mean'])
                           .fillna(self.global_mean))
        return pred.clip(lower=0).values


run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='NBEATS_Final_Pipeline',
                 job_type='final_pipeline',
                 config={**BEST_CFG, 'pred_len': PRED_LEN_TEST, 'exogenous': 'IsHoliday'})
run.summary['val_wmae'] = best_val

mu_f = wide.values.mean(axis=0)
sigma_f = np.clip(wide.values.std(axis=0), 1.0, None)
norm_full = (wide.values - mu_f) / sigma_f
full_holiday = holiday_by_date.reindex(wide.index).values

final_model, losses = train_nbeats(norm_full, full_holiday, pred_len=PRED_LEN_TEST, **BEST_CFG,
                                    log_fn=lambda ep, l: run.log({'epoch': ep, 'train_mse': l}))

pred = forecast_next(final_model, norm_full, test_holiday_future, BEST_CFG['seq_len']) * sigma_f + mu_f
pred = np.clip(pred, 0, None)

pred_df = pd.DataFrame(pred, index=test_dates, columns=wide.columns)
forecast_long = pred_df.stack([0, 1]).rename('pred').reset_index()
forecast_long.columns = ['Date', 'Store', 'Dept', 'pred']

sd_mean = train.groupby(['Store', 'Dept'])['Weekly_Sales'].mean().rename('SD_Mean').reset_index()
dept_mean = train.groupby('Dept')['Weekly_Sales'].mean().rename('Dept_Mean').reset_index()
global_mean = float(train['Weekly_Sales'].mean())

pipeline = NBEATSForecastPipeline(forecast_long, sd_mean, dept_mean, global_mean)

val_score = best_val
with open('nbeats_pipeline.pkl', 'wb') as f:
    cloudpickle.dump(pipeline, f)
art = wandb.Artifact('nbeats_pipeline', type='model')
art.add_file('nbeats_pipeline.pkl')
run.log_artifact(art)
run.finish()
print('Pipeline saved. val_wmae:', val_score)

test horizon: 39 weeks


  epoch  10: train MSE 0.6608
  epoch  20: train MSE 0.5930
  epoch  30: train MSE 0.5769
  epoch  40: train MSE 0.5662
  epoch  50: train MSE 0.5575
  epoch  60: train MSE 0.5504


/tmp/ipykernel_552/2663555169.py:46: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  forecast_long = pred_df.stack([0, 1]).rename('pred').reset_index()


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train_mse,█▆▅▄▄▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,59
train_mse,0.5504
val_wmae,1463.00219


Pipeline saved. val_wmae: 1463.00219


In [15]:
import json, os

reg_path = '/content/drive/MyDrive/ML Final/best_runs.json'
info = json.load(open(reg_path)) if os.path.exists(reg_path) else {}
info['NBEATS'] = {
    'artifact': f'{WANDB_PROJECT}/nbeats_pipeline:latest',
    'val_wmae': float(val_score),
}
with open(reg_path, 'w') as f:
    json.dump(info, f, indent=2)
info

{'XGBoost': {'artifact': 'ML-Final/xgboost_pipeline:latest',
  'val_wmae': 1451.361784955096},
 'LightGBM': {'artifact': 'ML-Final/lightgbm_pipeline:latest',
  'val_wmae': 1508.4106169389534},
 'DLinear': {'artifact': 'ML-Final/dlinear_pipeline:latest',
  'val_wmae': 1602.6865936618267},
 'NBEATS': {'artifact': 'ML-Final/nbeats_pipeline:latest',
  'val_wmae': 1463.00219},
 'PatchTST': {'artifact': 'ML-Final/patchtst_pipeline:latest',
  'val_wmae': 1554.1872775576571},
 'Prophet': {'artifact': 'ML-Final/prophet_pipeline:latest',
  'val_wmae': 1831.404616769137},
 'ARIMA': {'artifact': 'ML-Final/arima_pipeline:latest',
  'val_wmae': 1914.1948218719908},
 'SARIMA': {'artifact': 'ML-Final/sarima_pipeline:latest',
  'val_wmae': 2137.2593427953866}}

In [16]:
api = wandb.Api()
art = api.artifact(f'{WANDB_PROJECT}/nbeats_pipeline:latest')
art_dir = art.download()
with open(os.path.join(art_dir, 'nbeats_pipeline.pkl'), 'rb') as f:
    loaded = cloudpickle.load(f)
print(loaded.predict(test[RAW_COLS].head())[:5])

wandb:   1 of 1 files downloaded.  


[25933.44906129 20854.32460162 20438.6025291  23689.80491562
 23482.4375132 ]


In [ ]:
import json
with open('/content/drive/MyDrive/ML Final/best_runs.json') as f:
    print(json.load(f))

{'XGBoost': {'artifact': 'ML-Final/xgboost_pipeline:latest', 'val_wmae': 1451.361784955096}, 'LightGBM': {'artifact': 'ML-Final/lightgbm_pipeline:latest', 'val_wmae': 1508.4106169389534}, 'DLinear': {'artifact': 'ML-Final/dlinear_pipeline:latest', 'val_wmae': 1602.6865936618267}, 'NBEATS': {'artifact': 'ML-Final/nbeats_pipeline:latest', 'val_wmae': 1476.8615020275404}}


Experimental Runs

In [12]:
tr_wide = wide[wide.index < VAL_CUTOFF]
mu = tr_wide.values.mean(axis=0)
sigma = np.clip(tr_wide.values.std(axis=0), 1.0, None)
norm_tr = (tr_wide.values - mu) / sigma

tr_holiday = holiday_by_date.reindex(tr_wide.index).values
val_holiday_future = holiday_by_date.reindex(val_dates).values.astype(np.float32)

best_val, BEST_CFG = 1476.8615020275404, dict(seq_len=52, n_blocks=3, layer_width=128, epochs=60, lr=1e-3)

EXTRA_CANDIDATES = [
    dict(seq_len=26, n_blocks=3, layer_width=128, epochs=60, lr=1e-3),
    dict(seq_len=104, n_blocks=3, layer_width=128, epochs=60, lr=1e-3),
    dict(seq_len=52, n_blocks=2, layer_width=64, epochs=60, lr=1e-3),
    dict(seq_len=52, n_blocks=3, layer_width=128, epochs=60, lr=2e-3),
    dict(seq_len=52, n_blocks=3, layer_width=128, epochs=100, lr=1e-3),
    dict(seq_len=52, n_blocks=3, layer_width=128, epochs=60, lr=1e-3, n_layers=2),
    dict(seq_len=52, n_blocks=5, layer_width=128, epochs=60, lr=1e-3),
    dict(seq_len=52, n_blocks=3, layer_width=128, epochs=60, lr=1e-3, batch_size=32),
    dict(seq_len=39, n_blocks=3, layer_width=128, epochs=60, lr=1e-3),
    dict(seq_len=52, n_blocks=3, layer_width=192, epochs=60, lr=1e-3),
    dict(seq_len=52, n_blocks=6, layer_width=128, epochs=60, lr=1e-3),
    dict(seq_len=52, n_blocks=3, layer_width=128, epochs=60, lr=1e-3, n_layers=5),
]

for j, cfg in enumerate(EXTRA_CANDIDATES):
    i = 4 + j  # continue numbering after candidates 0-3
    run = wandb.init(project=WANDB_PROJECT, group=GROUP, name=f'NBEATS_candidate_{i}',
                     job_type='hyperparam_search', reinit=True,
                     config={**cfg, 'pred_len': PRED_LEN_VAL, 'exogenous': 'IsHoliday',
                             'val_scheme': f'train < {VAL_CUTOFF}, predict {PRED_LEN_VAL} weeks ahead',
                             'sweep_phase': 'extended'})
    print(f'candidate {i}: {cfg}')
    model, losses = train_nbeats(norm_tr, tr_holiday, pred_len=PRED_LEN_VAL, **cfg,
                                  log_fn=lambda ep, l: run.log({'epoch': ep, 'train_mse': l}))
    pred = forecast_next(model, norm_tr, val_holiday_future, cfg['seq_len']) * sigma + mu
    score = float(matrix_val_wmae(pred))
    run.summary['val_wmae'] = score
    run.finish()
    print(f'  -> val WMAE {score:,.2f}')
    if score < best_val:
        best_val, BEST_CFG = score, cfg

print('\nOverall best after extended search:', BEST_CFG, f'-> {best_val:,.2f}')

candidate 4: {'seq_len': 26, 'n_blocks': 3, 'layer_width': 128, 'epochs': 60, 'lr': 0.001}
  epoch  10: train MSE 0.9089
  epoch  20: train MSE 0.8534
  epoch  30: train MSE 0.7779
  epoch  40: train MSE 0.7322
  epoch  50: train MSE 0.7204
  epoch  60: train MSE 0.7338


/tmp/ipykernel_552/1391165103.py:66: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  pred_long = pred_df.stack([0, 1]).rename('pred').reset_index()


epoch,▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███
train_mse,█▇▇▆▆▆▆▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
epoch,59
train_mse,0.73385
val_wmae,1770.63147


  -> val WMAE 1,770.63


candidate 5: {'seq_len': 104, 'n_blocks': 3, 'layer_width': 128, 'epochs': 60, 'lr': 0.001}
  epoch  10: train MSE 0.4963
  epoch  20: train MSE 0.4332
  epoch  30: train MSE 0.4034
  epoch  40: train MSE 0.3863
  epoch  50: train MSE 0.3860
  epoch  60: train MSE 0.3670


...

  -> val WMAE 1,498.24


candidate 15: {'seq_len': 52, 'n_blocks': 3, 'layer_width': 128, 'epochs': 60, 'lr': 0.001, 'n_layers': 5}
  epoch  10: train MSE 0.6538
  epoch  20: train MSE 0.5792
  epoch  30: train MSE 0.5474
  epoch  40: train MSE 0.5333
  epoch  50: train MSE 0.5162
  epoch  60: train MSE 0.5126


/tmp/ipykernel_552/1391165103.py:66: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  pred_long = pred_df.stack([0, 1]).rename('pred').reset_index()


epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇███
train_mse,█▇▆▅▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,59
train_mse,0.51256
val_wmae,1509.67443


  -> val WMAE 1,509.67

Overall best after extended search: {'seq_len': 52, 'n_blocks': 3, 'layer_width': 128, 'epochs': 60, 'lr': 0.001, 'n_layers': 2} -> 1,463.00


In [13]:
best_val = 1463.00219
BEST_CFG = dict(seq_len=52, n_blocks=3, layer_width=128, epochs=60, lr=1e-3, n_layers=2)